# Fish gill shape sweep

Fish gills at fixed size with the shape swept in three parameters: `n_rods` filaments along the arch, `l_rod` center nodes per filament, and `lamellae` lamellae per center node, so

`N = n_rods * (1 + l_rod * (1 + lamellae))`

and N runs over the powers of two from 16 to 1024 (146 shapes). Every fish graph is a **tree**, E = N - 1, so each shape is compared with 100 random trees of the same N.

**The sweep is 3-D but each size is 2-D.** The product above pins the third parameter once N and two others are fixed, so a size is a grid over (`n_rods`, `l_rod`) with `lamellae` implied. That grid is ragged, since `l_rod` must divide `N / n_rods - 1`: at N = 1024 only 38 of the 200 (`n_rods`, `l_rod`) pairs exist. The figures below therefore plot the shapes that exist as points rather than filling a matrix.

Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.

Mean times are conditional on fixation. `fixation_prob` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm, Normalize

from moran_graphs import theory
from moran_graphs.plots import COLORS, LABELS, lines, load, print_batch, print_run_report
from moran_graphs.PopulationGraph import PopulationGraph

Six of the 16 shapes at N = 64, from the bushiest to the thinnest: the arch runs vertically, filaments to the right, lamellae fanned out from each center node.
`lamellae` sets the largest degree (a center node holds `lamellae` lamellae plus its two neighbors), so the first panel is the star K_{1,63} and the last is the path P_64.

In [ ]:
shapes = [(1, 1, 62), (1, 3, 20), (4, 1, 14), (8, 1, 6), (16, 1, 2), (1, 63, 0)]
fig, axes = plt.subplots(1, len(shapes), figsize=(15, 2.8), layout="constrained")
for ax, (n, l, f) in zip(axes, shapes):
    gill = PopulationGraph.fish_graph(n_rods=n, l_rod=l, lamellae=f).graph
    nx.draw(gill, nx.get_node_attributes(gill, "pos"), ax=ax, node_size=8, width=0.5, node_color=COLORS["Fish gill"])
    ax.set_title(f"n_rods = {n}, l_rod = {l}\nlamellae = {f}", fontsize=8)

## Configuration

The only cell to edit.

In [ ]:
BATCHES = ["2026-10-25-fish-shape-sweep"]
R_VALUES = [1.1]  # the figures need exactly one

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

`load` also divides every graph's fixation probability and mean fixation time by the mean random graph with the same N, E and r (`*_over_random`), as in `versus_random`. Here that reference is a random tree, since E = N - 1 for every fish graph.

In [ ]:
results, hists = load(BATCHES)
results = results[results["r"].isin(R_VALUES)]

random = results[results["category"] == "Random"]
fish = results[results["category"] == "Fish gill"]
recipe = pd.json_normalize(fish["kwargs"].map(json.loads).tolist()).set_index(fish.index)
fish = fish.assign(n_rods=recipe["n_rods"], l_rod=recipe["l_rod"], lamellae=recipe["lamellae"])
assert (fish["n_edges"] == fish["n_nodes"] - 1).all()  # every fish graph is a tree, so the random matches are trees
print(f"{len(fish)} fish shapes over {fish['n_nodes'].nunique()} sizes, {len(random)} random trees")
fish.groupby("n_nodes").size().rename("shapes").to_frame().T

## Monte Carlo precision

How well each (graph, r) cell is measured: runs, fixed runs, SE and the relative 95% CI half-width of both metrics, against N. The dashed line is the target half-width `stage_two_plan` sizes n for. Fixed runs set the precision of both metrics, so cells with a small fixation probability are the noisiest.

In [ ]:
from moran_graphs.plots import precision

fig = precision(results)

Fixation-time distributions per graph type and N (left) against the matched random cloud (right), one figure per r. Narrow, untailed distributions need few fixed runs for a precise mean; the numbers at the bottom are the median CV. Where several graphs share a type and N (shape sweeps), their runs are pooled, so the left half also includes differences between shapes.

In [ ]:
from moran_graphs.plots import time_violins

for r in sorted(results["r"].unique()):
    fig = time_violins(results, hists, r)

How much of each random cloud's spread is simulation noise (Monte Carlo variance over the observed variance between its graphs). Near 0 the middle-95% band measures structure; near 1 the band, and the stands-out test built on it, is mostly noise.

In [ ]:
from moran_graphs.plots import random_noise_share

fig = random_noise_share(results)

Stage-two batches only: each cell's estimate against its independent pilot estimate as a z-score. About 5% of cells should fall beyond 1.96; many more means the engine, the seeding or the SE is wrong. Draws nothing for batches without a pilot.

In [ ]:
from moran_graphs.plots import pilot_agreement

fig = pilot_agreement(results)

In [ ]:
fish.columns

In [ ]:
fish[["n_rods", "l_rod", "lamellae"]].drop_duplicates().sort_values(
    ["n_rods", "l_rod", "lamellae"]
).reset_index(drop=True)

## Check: one rod of one node is the star

With `n_rods` = 1 and `l_rod` = 1 the graph is an arch node, one center node, and `lamellae` = N - 2 lamellae on it. The center node touches everything, so this is exactly the star K_{1,N-1}, whose fixation probability is known (`theory.star`, Bd).
The simulation should agree within its 95% Wilson interval; that is 7 comparisons, one per size. `theory.star` gives no fixation time, so only `fixation_prob` is checked.

In [ ]:
stars = fish[(fish["n_rods"] == 1) & (fish["l_rod"] == 1)].sort_values("n_nodes")
check = stars[["n_nodes", "lamellae", "fixation_prob_lo", "fixation_prob", "fixation_prob_hi"]].assign(
    fixation_prob_exact=[theory.star(N, r)[0] for N, r in zip(stars["n_nodes"], stars["r"])])
check["fixation_prob_ok"] = check["fixation_prob_exact"].between(check["fixation_prob_lo"], check["fixation_prob_hi"])
check.round(4)

## The shape space, one panel per size

One point per shape at `n_rods` x `l_rod`, both on a log scale, with `lamellae` = `N` / `n_rods` / `l_rod` - 1 / `l_rod` fixed by the two. Colors are the shape over the mean random tree of the same N: red above 1, blue below, white at the random tree, so the panels say which shapes beat a random tree of the same size rather than which are big.

Fixation probability is on a linear color scale and time on a log one (so x2 and x1/2 read as equally strong), as in the avian sweep. Empty regions are shapes the product `N = n_rods * (1 + l_rod * (1 + lamellae))` does not allow.

In [ ]:
def shape_panels(df, y, title, scale):
    values = df[y]
    if scale == "log":
        extreme = max(values.max(), 1 / values.min())
        norm = LogNorm(1 / extreme, extreme)
    else:
        spread = np.abs(values - 1).max()
        norm = Normalize(1 - spread, 1 + spread)

    sizes = sorted(df["n_nodes"].unique())
    fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharex=True, sharey=True, layout="constrained")
    for ax, N in zip(axes.flat, sizes):
        part = df[df["n_nodes"] == N]
        dots = ax.scatter(part["n_rods"], part["l_rod"], c=part[y], cmap="RdBu_r", norm=norm,
                          s=110, edgecolors="0.3", linewidths=0.4)
        ax.set(xscale="log", yscale="log", title=f"N = {N}", xlabel="n_rods", ylabel="l_rod")
    axes.flat[len(sizes)].axis("off")
    fig.colorbar(dots, ax=axes, label=title)
    fig.suptitle(f"{title}, r = {R_VALUES[0]}")


shape_panels(fish, "fixation_prob_over_random", LABELS["fixation_prob_over_random"], "linear")
shape_panels(fish, "mean_fixation_time_over_random", LABELS["mean_fixation_time_over_random"], "log")

## The three edges of the shape space

Each edge holds one parameter at its smallest value and sweeps the other two against each other, which makes a clean line per size:

- **`l_rod` = 1**: every filament is a single center node carrying its lamellae, so the arch gains rods as the lamellae thin out. Left end is the star, right end a caterpillar.
- **`lamellae` = 0**: no lamellae, so the graph is `n_rods` bare paths on an arch, a comb. Left end is the path P_N.
- **`n_rods` = 1**: a single filament, trading its length against its lamellae.

Top row is the shape over the mean random tree of the same size, bottom row the raw value (time on a log scale). Grey is each random tree over that same mean: the spread a shape has to leave to stand out. Error bars are 95% intervals (Wilson for `fixation_prob`, 1.96 SE for time), raw row only, since on the ratio row the random spread is the reference.

In [ ]:
edges = (("l_rod == 1", "n_rods"), ("lamellae == 0", "n_rods"), ("n_rods == 1", "l_rod"))
rows = (("fixation_prob_over_random", LABELS["fixation_prob_over_random"], "linear"),
        ("fixation_prob", LABELS["fixation_prob"], "linear"),
        ("mean_fixation_time_over_random", LABELS["mean_fixation_time_over_random"], "log"),
        ("mean_fixation_time", LABELS["mean_fixation_time"], "log"))

fig, axes = plt.subplots(len(rows), len(edges), figsize=(16, 16), layout="constrained")
for column, (edge, x) in zip(axes.T, edges):
    column[0].set_title(f"{edge}   (r = {R_VALUES[0]})")
    for ax, (y, ylabel, yscale) in zip(column, rows):
        if y.endswith("_over_random"):
            ax.axhline(1, color="0.2", lw=1)
        else:
            ax.scatter(random["n_nodes"], random[y], s=8, alpha=0.5, linewidths=0, color=COLORS["Random"],
                       label="random trees, same N")
        lines(ax, fish.query(edge), x, y, by="n_nodes")
        ax.set(xscale="log", yscale=yscale, ylabel=ylabel)
fig.legend(*axes[1, 0].get_legend_handles_labels(), loc="outside right center", frameon=False, fontsize=9)

## One panel per size

Each panel is one N, so it holds every shape of that size, and the grey band is the middle 95% of the 100 random trees of that N (on the ratio rows, each tree over the mean random tree, as in Load). One line per `n_rods` with `lamellae` on x, so `l_rod` = (N / `n_rods` - 1) / (1 + `lamellae`) is implied: a line runs from bare paths on the arch (`lamellae` = 0) to one center node per filament (`l_rod` = 1). A color is the same `n_rods` in every panel. A line can span shapes the sweep does not have: 127 is prime, so at N = 128 `n_rods` = 1 holds only the path and the star.

Every axis is an argument of `line_panels`: `x` with its `xscale`, `lines_by`, each row as (column, label, scale), and `panels_by`; the grey band is drawn only when the panels are sizes. Each panel has its own x range, since `lamellae` runs up to N - 2. Error bars as in the edge figure, on the raw rows only.

In [ ]:
from moran_graphs.plots import _error  # the error bars lines() draws


def line_panels(df, x, xscale, lines_by, rows, panels_by="n_nodes"):
    # one color per value of lines_by across all panels, so a value keeps its color from N to N
    values = sorted(df[lines_by].unique())
    colors = dict(zip(values, plt.get_cmap("viridis")(np.linspace(0, 0.9, len(values)))))
    panels = sorted(df[panels_by].unique())
    fig, axes = plt.subplots(len(rows), len(panels), figsize=(3.2 * len(panels), 3.4 * len(rows)),
                             sharey="row", squeeze=False, layout="constrained")
    handles = {}
    for row, (y, ylabel, yscale) in zip(axes, rows):
        for ax, panel in zip(row, panels):
            if panels_by == "n_nodes":
                spread = random.loc[random["n_nodes"] == panel, y].quantile([0.025, 0.975])
                ax.axhspan(*spread, color="0.88", zorder=0)
            if y.endswith("_over_random"):
                ax.axhline(1, color="0.2", lw=1)
            for value, part in df[df[panels_by] == panel].groupby(lines_by):
                part = part.sort_values(x)
                handles[value] = ax.errorbar(part[x], part[y], yerr=_error(part, y), fmt="o-", ms=4, lw=1.5,
                                             capsize=0, color=colors[value])
            ax.set(xscale=xscale, yscale=yscale)
        row[0].set_ylabel(ylabel)
    for ax, panel in zip(axes[0], panels):
        ax.set_title(f"{panels_by} = {panel}")
    for ax in axes[-1]:
        ax.set_xlabel(x)
    fig.legend([handles[value] for value in values], [f"{lines_by} = {value}" for value in values],
               loc="outside right center", frameon=False, fontsize=8, ncols=1 + len(values) // 20)
    fig.suptitle(f"One line per {lines_by}, r = {R_VALUES[0]}")


line_panels(fish, x="n_nodes", lines_by="lamellae", xscale="linear", panels_by="n_rods",
            rows=(("fixation_prob_over_random", LABELS["fixation_prob_over_random"], "linear"),
                  ("fixation_prob", LABELS["fixation_prob"], "linear"),))

line_panels(fish, x="n_nodes", lines_by="lamellae", xscale="linear", panels_by="n_rods",
            rows=(("mean_fixation_time_over_random", LABELS["mean_fixation_time_over_random"], "log"),
                  ("mean_fixation_time", LABELS["mean_fixation_time"], "log")))

In [ ]:
fish.groupby(["n_rods", "l_rod", "n_nodes"]).size().describe()   # all 1 means single-point lines
fish[["mean_fixation_time_over_random", "fixation_prob"]].isna().mean()

In [ ]:
import seaborn as sns
sns.relplot(fish, kind="scatter", row="n_rods", col="n_nodes",
            x="mean_fixation_time_over_random", y="fixation_prob", hue="lamellae",
            facet_kws={"sharey": True})

In [ ]:

line_panels(fish, x="n_nodes", lines_by="lamellae", xscale="linear", panels_by="n_rods",
            rows=(("mean_fixation_time_over_random", LABELS["mean_fixation_time_over_random"], "log"),
                  ("mean_fixation_time", LABELS["mean_fixation_time"], "log")))

## What the shape is really doing

Across all 146 shapes the two ratios line up with the largest degree, which is `lamellae` + 2 at any center node that carries lamellae (`lamellae` + 1 when the filament is a single node). A gill with one fat hub amplifies and is slow; a thin comb of paths sits at or below the random tree.

Left: against the largest degree as a fraction of N, which puts every size on one axis. Right: against the mean shortest path, the same shapes seen as depth rather than hub strength. Grey is the random trees of the same sizes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), layout="constrained")
for column, x, xlabel, xscale in ((0, "max_degree", "largest degree / N", "log"),
                                  (1, "average_shortest_path_length", "mean shortest path", "log")):
    for row, (y, ylabel, yscale) in enumerate((("fixation_prob_over_random", LABELS["fixation_prob_over_random"], "linear"),
                                               ("mean_fixation_time_over_random", LABELS["mean_fixation_time_over_random"], "log"))):
        ax = axes[row, column]
        scale = random["n_nodes"] if x == "max_degree" else 1
        ax.scatter(random[x] / scale, random[y], s=8, alpha=0.4, linewidths=0, color=COLORS["Random"],
                   label="random trees")
        dots = ax.scatter(fish[x] / (fish["n_nodes"] if x == "max_degree" else 1), fish[y], c=fish["n_nodes"],
                          norm=LogNorm(), cmap="viridis", s=26, label="fish shapes")
        ax.axhline(1, color="0.2", lw=1)
        ax.set(xscale=xscale, yscale=yscale, xlabel=xlabel, ylabel=ylabel)
fig.colorbar(dots, ax=axes, label="N")
axes[0, 0].legend(frameon=False, fontsize=9)
fig.suptitle(f"Every fish shape against its hub strength and its depth, r = {R_VALUES[0]}")

spearman = fish[["max_degree", "average_shortest_path_length", "degree_std", "diameter"]].corrwith(
    fish["fixation_prob_over_random"], method="spearman")
spearman.rename("spearman with fixation_prob_over_random").to_frame().round(3)